In [1]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

In [6]:
CRS = 'EPSG:4326'
PATH_DF_JOINED_ACLED_EVENTS = 
PATH_GDF_JOINED_ACLED_EVENTS

### Data import

In [40]:
PATH_TO_ACLED_EVENTS = '../data/raw/Middle-East_aggregated_data_up_to_week_of-2026-10-31.csv'
PATH_TO_ADMIN_BOUNDARIES = '../data/raw/pse_admin_boundaries.shp/pse_admin2.shp'

df_events = pd.read_csv(PATH_TO_ACLED_EVENTS)
gdf_boundaries = gpd.read_file(PATH_TO_ADMIN_BOUNDARIES)

In [34]:
#df_events.head()

In [35]:
#gdf_boundaries.head()

### Preprocessing

In [41]:
# strip whitespaces
string_cols = ['REGION', 'COUNTRY', 'ADMIN1', 'EVENT_TYPE', 'SUB_EVENT_TYPE', 'DISORDER_TYPE']
df_events[string_cols] = df_events[string_cols].apply(lambda x: x.str.strip())

# format date columns
df_events['WEEK'] = pd.to_datetime(df_events['WEEK'])

# create date grouping columns
df_events['YEAR'] = df_events['WEEK'].dt.year
df_events['YEARMO'] = df_events['WEEK'].dt.to_period('M').astype(str)
df_events['YEAR_WEEK'] = df_events['WEEK'].dt.to_period('W').astype(str)

mask_palestine_events = df_events.COUNTRY == 'Palestine'
df_events = df_events[mask_palestine_events]

### Geoprocessing

In [42]:
# convert event coordinates into spatial points
gdf_events = gpd.GeoDataFrame(
    df_events,
    geometry=gpd.points_from_xy(
        df_events['CENTROID_LONGITUDE'],
        df_events['CENTROID_LATITUDE']
    ),
    crs=CRS
)

In [43]:
# check and align CRSs
print("Events CRS:", gdf_events.crs)
print("Admin CRS:", gdf_boundaries.crs)
gdf_events = gdf_events.to_crs(gdf_boundaries.crs) # failsafe -- convert events CRS to admin boundaries CRS if not already aligned
assert(gdf_events.crs == gdf_boundaries.crs)
print('Confirmed CRSs are aligned')

Events CRS: EPSG:4326
Admin CRS: EPSG:4326
Confirmed CRSs are aligned


In [45]:
# perform spatial join (assign event coordinates to an administrative boundary)
gdf_events_joined = gpd.sjoin(
    gdf_events,
    gdf_boundaries[['adm1_name', 'adm1_pcode', 'adm2_name', 'adm2_pcode', 'geometry']],
    how='left',
    predicate='within'
)

In [26]:
#gdf_events_joined.head()

,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,...,CENTROID_LONGITUDE,YEAR,YEARMO,YEAR_WEEK,geometry,index_right,adm1_name,adm1_pcode,adm2_name,adm2_pcode
58647,2016-04-09,Middle East,Palestine,Gaza Strip,Battles,Armed clash,1,0,171407.0,Political violence,...,34.3577,2016,2016-04,2016-04-04/2016-04-10,POINT (34.3577 31.407),13,Gaza Strip,PS02,Deir Al-Balah,PS0265
58648,2016-04-30,Middle East,Palestine,Gaza Strip,Battles,Armed clash,4,0,169999.0,Political violence,...,34.3577,2016,2016-04,2016-04-25/2016-05-01,POINT (34.3577 31.407),13,Gaza Strip,PS02,Deir Al-Balah,PS0265
58649,2016-05-07,Middle East,Palestine,Gaza Strip,Battles,Armed clash,2,0,207651.0,Political violence,...,34.3577,2016,2016-05,2016-05-02/2016-05-08,POINT (34.3577 31.407),13,Gaza Strip,PS02,Deir Al-Balah,PS0265
58650,2016-09-03,Middle East,Palestine,Gaza Strip,Battles,Armed clash,1,0,30699.0,Political violence,...,34.3577,2016,2016-09,2016-08-29/2016-09-04,POINT (34.3577 31.407),13,Gaza Strip,PS02,Deir Al-Balah,PS0265
58651,2016-12-03,Middle East,Palestine,Gaza Strip,Battles,Armed clash,1,0,13502.0,Political violence,...,34.3577,2016,2016-12,2016-11-28/2016-12-04,POINT (34.3577 31.407),13,Gaza Strip,PS02,Deir Al-Balah,PS0265


In [28]:
print("Total events:", len(gdf_events_joined))
print("Matched events:", gdf_events_joined['adm1_name'].notna().sum())
print("Unmatched events:", gdf_events_joined['adm1_name'].isna().sum())

Total events: 7849
Matched events: 7849
Unmatched events: 0


In [31]:
unmatched = gdf_events_joined[gdf_events_joined['ADMIN1'].isna()]
if len(unmatched) > 0:
    display(unmatched.head())
else:
    print('All events matched successfully!')

All events matched successfully!


In [32]:
 gdf_events_joined['adm1_name'].value_counts()

adm1_name
West Bank     4690
Gaza Strip    3159
Name: count, dtype: int64

In [33]:
 gdf_events_joined['adm2_name'].value_counts()

adm2_name
Ramallah         4690
Deir Al-Balah    3159
Name: count, dtype: int64

#### TODO: Why are all events confined to one governorate per administrative boundary?

### Governorate Aggregates

#### Total Events

In [47]:
#gdf_events_joined.head()

#### Total Events (by type)

#### Total Fatalities

### Export Results